In [57]:
%pip install supabase python-dotenv pandas numpy scipy scikit-learn PySastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [58]:
# =========================================================
# CELL 1 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import json
import scipy.sparse as sp
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from datetime import datetime, timezone
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY") or os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

SOURCE_TABLE = "cleaned_papers_results"
EVAL_TABLE = "evaluation_precision_at_k"

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
tfidf_dir = os.path.join(base_dir, "data", "tfidf")
eval_dir = os.path.join(base_dir, "data", "evaluation")

os.makedirs(eval_dir, exist_ok=True)

print("✅ Import dan koneksi Supabase berhasil")

✅ Import dan koneksi Supabase berhasil


In [59]:
# =========================================================
# CELL 2 - LOAD FILE VSM HASIL TF-IDF
# =========================================================
tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, "tfidf_matrix.npz"))

with open(os.path.join(tfidf_dir, "tfidf_terms.json"), "r", encoding="utf-8") as file:
    terms = json.load(file)

with open(os.path.join(tfidf_dir, "tfidf_doc_ids.json"), "r", encoding="utf-8") as file:
    doc_ids = json.load(file)

with open(os.path.join(tfidf_dir, "idf_scores.json"), "r", encoding="utf-8") as file:
    idf_scores = json.load(file)

tfidf_documents = pd.read_csv(os.path.join(tfidf_dir, "tfidf_documents.csv"))

doc_ids = [int(doc_id) for doc_id in doc_ids]

print("✅ File VSM berhasil dimuat")
print("Matrix TF-IDF:", tfidf_matrix.shape)
print("Jumlah terms:", len(terms))
print("Jumlah doc_ids:", len(doc_ids))

✅ File VSM berhasil dimuat
Matrix TF-IDF: (200, 1723)
Jumlah terms: 1723
Jumlah doc_ids: 200


In [60]:
# =========================================================
# CELL 3 - LOAD METADATA ARTIKEL DARI SUPABASE
# =========================================================
all_data = []
batch_size = 1000
offset = 0

selected_columns = """
id,
title,
abstract,
authors,
year,
source,
category,
pdf_url,
url,
scrape_status
"""

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select(selected_columns)
        .range(offset, offset + batch_size - 1)
        .execute()
    )

    batch = response.data or []

    if not batch:
        break

    all_data.extend(batch)

    if len(batch) < batch_size:
        break

    offset += batch_size

metadata_df = pd.DataFrame(all_data)

if metadata_df.empty:
    raise ValueError("❌ Data cleaned_papers_results kosong.")

metadata_df["id"] = metadata_df["id"].astype("int64")

doc_index = pd.DataFrame({"id": doc_ids})
doc_index = doc_index.merge(metadata_df, on="id", how="left")
doc_index = doc_index.merge(
    tfidf_documents[["id", "document_text"]],
    on="id",
    how="left"
)

for col in ["title", "abstract", "authors", "source", "category", "pdf_url", "url", "scrape_status", "document_text"]:
    if col in doc_index.columns:
        doc_index[col] = doc_index[col].fillna("")

if tfidf_matrix.shape[0] != len(doc_index):
    raise ValueError("❌ Jumlah matrix TF-IDF tidak sama dengan jumlah dokumen.")

print("✅ Metadata artikel siap")
print("Jumlah dokumen:", len(doc_index))
doc_index.head(3)

✅ Metadata artikel siap
Jumlah dokumen: 200


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status,document_text
0,1,Machine learning for microbiologists,… how to evaluate a machine learning model and...,"F Asnicar, AM Thomas, A Passerini…",2024,Nature Reviews …,machine learning,https://pmc.ncbi.nlm.nih.gov/articles/PMC11980...,https://www.nature.com/articles/s41579-023-009...,pdf_failed,machine learning microbiologists how evaluate ...
1,2,What is machine learning?,… that one can employ in machine learning (ML)...,J Bell,2022,Machine learning and the city: applications in …,machine learning,,https://onlinelibrary.wiley.com/doi/abs/10.100...,metadata_only,what machine learning one can employ machine l...
2,3,Amnesiac machine learning,… It gives EU residents the ability to request...,"L Graves, V Nagisetty, V Ganesh",2021,… of the AAAI conference on artificial …,machine learning,https://ojs.aaai.org/index.php/AAAI/article/do...,https://ojs.aaai.org/index.php/AAAI/article/vi...,pdf_downloaded,amnesiac machine learning gives eu residents a...


In [61]:
# =========================================================
# CELL 4 - STOPWORDS + STEMMER
# =========================================================
stop_words = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan", "pada",
    "adalah", "dalam", "atau", "sebagai", "oleh", "karena", "terhadap", "akan",
    "dapat", "lebih", "juga", "tidak", "ada", "antara", "para", "saat", "telah",
    "menjadi", "yaitu", "yakni", "bahwa", "sebuah", "suatu", "agar", "bagi",
    "tanpa", "setelah", "sebelum", "hingga", "maka", "namun", "serta",
    "the", "of", "and", "in", "to", "for", "a", "an", "is", "are", "on", "by",
    "with", "as", "at", "from", "or", "this", "that", "be", "it", "was", "were"
}

stemmer = StemmerFactory().create_stemmer()

print("✅ Stopwords dan stemmer siap")

✅ Stopwords dan stemmer siap


In [62]:
# =========================================================
# CELL 5 - FUNGSI QUERY, COSINE, DAN SEARCH
# =========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_query(query):
    cleaned = clean_text(query)
    tokens = cleaned.split()
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

def build_query_vector(query_tokens):
    term_to_index = {term: index for index, term in enumerate(terms)}
    query_vector = np.zeros(len(terms), dtype=float)

    if not query_tokens:
        return query_vector.reshape(1, -1)

    total_terms = len(query_tokens)
    term_counts = Counter(query_tokens)

    for term, count in term_counts.items():
        if term in term_to_index:
            tf = count / total_terms
            idf = float(idf_scores.get(term, 0))
            query_vector[term_to_index[term]] = tf * idf

    return query_vector.reshape(1, -1)

def count_occurrence(document_text, query_tokens):
    document_tokens = str(document_text).split()
    return sum(document_tokens.count(term) for term in query_tokens)

def interpret_similarity(score, threshold=0.3):
    if score > 0.5:
        return "Relevan Tinggi"
    if score > threshold:
        return "Relevan Sedang"
    return "Rendah"

def search(query, top_k=20, min_occurrence=0, threshold=0.3):
    query_tokens = preprocess_query(query)

    if not query_tokens:
        return pd.DataFrame()

    query_vector = build_query_vector(query_tokens)
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    results = doc_index.copy()
    results["similarity_score"] = scores
    results["occurrence"] = results["document_text"].apply(
        lambda text: count_occurrence(text, query_tokens)
    )

    results = results[results["similarity_score"] > 0].copy()

    if min_occurrence > 0:
        results = results[results["occurrence"] >= min_occurrence].copy()

    results = results.sort_values("similarity_score", ascending=False)
    results = results.head(top_k).reset_index(drop=True)

    results["rank"] = range(1, len(results) + 1)
    results["threshold_relevan"] = (results["similarity_score"] > threshold).astype(int)
    results["interpretation"] = results["similarity_score"].apply(
        lambda score: interpret_similarity(score, threshold)
    )

    return results

print("✅ Fungsi search evaluasi siap")

✅ Fungsi search evaluasi siap


In [63]:
# =========================================================
# CELL 6 - QUERY UJI EVALUASI
# K = 5, 10, 20 sesuai proposal
# =========================================================
queries_eval = [
    {"query": "machine learning", "kategori": "Machine Learning"},
    {"query": "deep learning", "kategori": "Machine Learning"},
    {"query": "data mining", "kategori": "Machine Learning"},
    {"query": "website application", "kategori": "Web Application"},
    {"query": "web system", "kategori": "Web Application"},
    {"query": "web application", "kategori": "Web Application"},
    {"query": "cyber security", "kategori": "Cyber Security"},
    {"query": "network security", "kategori": "Cyber Security"},
    {"query": "software security", "kategori": "Cyber Security"},
    {"query": "mobile application", "kategori": "Mobile Application"},
    {"query": "android application", "kategori": "Mobile Application"},
]

K_VALUES = [5, 10, 20]
MAX_K = max(K_VALUES)
THRESHOLD = 0.3

print("✅ Query evaluasi siap")

✅ Query evaluasi siap


In [64]:
# =========================================================
# CELL 7 - JALANKAN SEARCH UNTUK SEMUA QUERY
# =========================================================
all_results = {}
kemunculan_all = defaultdict(int)
kemunculan_kat = defaultdict(lambda: defaultdict(int))

for item in queries_eval:
    query = item["query"]
    kategori = item["kategori"]

    results = search(
        query=query,
        top_k=MAX_K,
        min_occurrence=0,
        threshold=THRESHOLD
    )

    all_results[query] = results

    for _, row in results.iterrows():
        kemunculan_all[row["title"]] += 1
        kemunculan_kat[kategori][row["title"]] += 1

print("✅ Semua query selesai diproses")

✅ Semua query selesai diproses


In [76]:
# =========================================================
# CELL 8 - BUAT TEMPLATE GROUND TRUTH
# Human judgment: isi relevan = 1 jika relevan, 0 jika tidak
# threshold_relevan hanya saran awal, bukan label final wajib
# =========================================================
rows = []

for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query]

    for _, row in results.iterrows():
        rows.append({
            "query": query,
            "kategori_query": kategori_query,
            "rank": int(row["rank"]),
            "article_id": int(row["id"]),
            "article_category": row["category"],
            "title": row["title"],
            "abstract": row["abstract"],
            "similarity_score": float(row["similarity_score"]),
            "threshold_relevan": int(row["threshold_relevan"]),
            "relevan": np.nan
        })

gt_template = pd.DataFrame(rows)

template_path = os.path.join(eval_dir, "ground_truth_template.csv")
labeled_path = os.path.join(eval_dir, "ground_truth_labeled.csv")

gt_template.to_csv(template_path, index=False)

print("✅ Template ground truth dibuat:", template_path)
gt_template.head(10)

✅ Template ground truth dibuat: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\ground_truth_template.csv


,query,kategori_query,rank,article_id,article_category,title,abstract,similarity_score,threshold_relevan,relevan
0,machine learning,Machine Learning,1,6,machine learning,Machine learning and deep learning: C. Janiesc...,… Deep learning is a machine learning concept ...,0.469064,1,NaN
1,machine learning,Machine Learning,2,37,machine learning,When machine learning meets privacy: A survey ...,… problems and machine learning is … machine l...,0.467015,1,NaN
2,machine learning,Machine Learning,3,17,machine learning,An overview of machine learning classification...,… statistical learning and pattern recognition...,0.446458,1,NaN
3,machine learning,Machine Learning,4,50,machine learning,"Impact of machine learning on management, heal...",… Machine learning and deep learning are two o...,0.445017,1,NaN
4,machine learning,Machine Learning,5,33,machine learning,Financial applications of machine learning: A ...,… of machine learning and deep learning in … o...,0.417879,1,NaN
5,machine learning,Machine Learning,6,36,machine learning,Scientific machine learning benchmarks,"… frameworks, computer architectures and machi...",0.416491,1,NaN
6,machine learning,Machine Learning,7,15,machine learning,Research on machine learning with algorithms a...,… This study provides a relatively systematic ...,0.409084,1,NaN
7,machine learning,Machine Learning,8,23,machine learning,Machine learning in chemical engineering: A pe...,… and application of machine learning with a f...,0.405659,1,NaN
8,machine learning,Machine Learning,9,21,machine learning,Machine learning and applications in microbiology,… Applying machine learning to address biologi...,0.386603,1,NaN
9,machine learning,Machine Learning,10,40,machine learning,Machine learning and deep learning application...,… Deep learning had been analysed and implemen...,0.382721,1,NaN


In [77]:
# =========================================================
# CELL 9 - INIT GROUND TRUTH LABELED
# Jika belum ada label manual, dibuat draft dari threshold > 0.3
# Untuk hasil final TA, edit file ground_truth_labeled.csv secara manual.
# =========================================================
if os.path.exists(labeled_path):
    gt_df = pd.read_csv(labeled_path)
    print("✅ Pakai ground_truth_labeled.csv yang sudah ada")
else:
    gt_df = gt_template.copy()
    gt_df["relevan"] = gt_df["threshold_relevan"].astype(int)
    gt_df.to_csv(labeled_path, index=False)
    print("⚠️ ground_truth_labeled.csv belum ada.")
    print("✅ Draft dibuat dari threshold > 0.3:", labeled_path)
    print("✍️ Untuk final TA, cek manual kolom relevan 0/1.")

gt_df["query"] = gt_df["query"].astype(str).str.lower().str.strip()
gt_df["article_id"] = gt_df["article_id"].astype("int64")
gt_df["relevan"] = gt_df["relevan"].fillna(0).astype(int)

invalid = set(gt_df["relevan"].unique()) - {0, 1}
if invalid:
    raise ValueError(f"Nilai relevan harus 0/1. Ditemukan: {invalid}")

print("✅ Ground truth siap")
gt_df.head(10)

⚠️ ground_truth_labeled.csv belum ada.
✅ Draft dibuat dari threshold > 0.3: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\ground_truth_labeled.csv
✍️ Untuk final TA, cek manual kolom relevan 0/1.
✅ Ground truth siap


,query,kategori_query,rank,article_id,article_category,title,abstract,similarity_score,threshold_relevan,relevan
0,machine learning,Machine Learning,1,6,machine learning,Machine learning and deep learning: C. Janiesc...,… Deep learning is a machine learning concept ...,0.469064,1,1
1,machine learning,Machine Learning,2,37,machine learning,When machine learning meets privacy: A survey ...,… problems and machine learning is … machine l...,0.467015,1,1
2,machine learning,Machine Learning,3,17,machine learning,An overview of machine learning classification...,… statistical learning and pattern recognition...,0.446458,1,1
3,machine learning,Machine Learning,4,50,machine learning,"Impact of machine learning on management, heal...",… Machine learning and deep learning are two o...,0.445017,1,1
4,machine learning,Machine Learning,5,33,machine learning,Financial applications of machine learning: A ...,… of machine learning and deep learning in … o...,0.417879,1,1
5,machine learning,Machine Learning,6,36,machine learning,Scientific machine learning benchmarks,"… frameworks, computer architectures and machi...",0.416491,1,1
6,machine learning,Machine Learning,7,15,machine learning,Research on machine learning with algorithms a...,… This study provides a relatively systematic ...,0.409084,1,1
7,machine learning,Machine Learning,8,23,machine learning,Machine learning in chemical engineering: A pe...,… and application of machine learning with a f...,0.405659,1,1
8,machine learning,Machine Learning,9,21,machine learning,Machine learning and applications in microbiology,… Applying machine learning to address biologi...,0.386603,1,1
9,machine learning,Machine Learning,10,40,machine learning,Machine learning and deep learning application...,… Deep learning had been analysed and implemen...,0.382721,1,1


In [78]:
# =========================================================
# CELL 10 - TABEL 1: HASIL PENCARIAN PER QUERY + LABEL
# =========================================================
tabel1_rows = []

for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query].copy()

    sub_gt = gt_df[gt_df["query"] == query][["article_id", "relevan"]].copy()
    sub_gt = sub_gt.rename(columns={"article_id": "id"})

    merged = results.merge(sub_gt, on="id", how="left")
    merged["relevan"] = merged["relevan"].fillna(0).astype(int)

    for _, row in merged.iterrows():
        tabel1_rows.append({
            "Query": query,
            "Kategori Query": kategori_query,
            "Rank": int(row["rank"]),
            "Article ID": int(row["id"]),
            "Kategori Artikel": row["category"],
            "Judul": row["title"],
            "Penulis": row["authors"],
            "Tahun": row["year"],
            "Source": row["source"],
            "Similarity Score": round(float(row["similarity_score"]), 6),
            "Threshold Relevan": "Ya" if row["threshold_relevan"] == 1 else "Tidak",
            "Relevan Human Judgment": "Ya" if row["relevan"] == 1 else "Tidak",
            "Occurrence": int(row["occurrence"]),
            "Interpretasi": row["interpretation"]
        })

tabel1_df = pd.DataFrame(tabel1_rows)

tabel1_path = os.path.join(eval_dir, "tabel1_hasil_pencarian.csv")
tabel1_df.to_csv(tabel1_path, index=False)

print("✅ Tabel 1 tersimpan:", tabel1_path)
tabel1_df.head(10)

✅ Tabel 1 tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\tabel1_hasil_pencarian.csv


,Query,Kategori Query,Rank,Article ID,Kategori Artikel,Judul,Penulis,Tahun,Source,Similarity Score,Threshold Relevan,Relevan Human Judgment,Occurrence,Interpretasi
0,machine learning,Machine Learning,1,6,machine learning,Machine learning and deep learning: C. Janiesc...,"C Janiesch, P Zschech, K Heinrich",2021,Electronic markets,0.469064,Ya,Ya,12,Relevan Sedang
1,machine learning,Machine Learning,2,37,machine learning,When machine learning meets privacy: A survey ...,"B Liu, M Ding, S Shaham, W Rahayu…",2021,ACM Computing …,0.467015,Ya,Ya,11,Relevan Sedang
2,machine learning,Machine Learning,3,17,machine learning,An overview of machine learning classification...,"AFAH Alnuaimi, THK Albaldawi",2024,BIO Web of Conferences,0.446458,Ya,Ya,10,Relevan Sedang
3,machine learning,Machine Learning,4,50,machine learning,"Impact of machine learning on management, heal...","H Pallathadka, M Mustafa, DT Sanchez…",2023,Materials Today …,0.445017,Ya,Ya,9,Relevan Sedang
4,machine learning,Machine Learning,5,33,machine learning,Financial applications of machine learning: A ...,"N Nazareth, YVR Reddy",2023,Expert Systems with Applications,0.417879,Ya,Ya,9,Relevan Sedang
5,machine learning,Machine Learning,6,36,machine learning,Scientific machine learning benchmarks,"J Thiyagalingam, M Shankar, G Fox, T Hey",2022,Nature Reviews Physics,0.416491,Ya,Ya,8,Relevan Sedang
6,machine learning,Machine Learning,7,15,machine learning,Research on machine learning with algorithms a...,"L Yu, X Zhao, J Huang, H Hu…",2023,Journal of Theory and …,0.409084,Ya,Ya,8,Relevan Sedang
7,machine learning,Machine Learning,8,23,machine learning,Machine learning in chemical engineering: A pe...,"AM Schweidtmann, E Esche, A Fischer…",2021,Chemie Ingenieur …,0.405659,Ya,Ya,8,Relevan Sedang
8,machine learning,Machine Learning,9,21,machine learning,Machine learning and applications in microbiology,"SJ Goodswen, JLN Barratt, PJ Kennedy…",2021,FEMS microbiology …,0.386603,Ya,Ya,8,Relevan Sedang
9,machine learning,Machine Learning,10,40,machine learning,Machine learning and deep learning application...,"N Sharma, R Sharma, N Jindal",2021,Global Transitions Proceedings,0.382721,Ya,Ya,10,Relevan Sedang


In [79]:
# =========================================================
# CELL 11 - TABEL 2: PRECISION@K
# Precision@K = jumlah relevan di Top K / K
# =========================================================
eval_rows = []

for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query].copy()

    sub_gt = gt_df[gt_df["query"] == query][["article_id", "relevan"]].copy()
    sub_gt = sub_gt.rename(columns={"article_id": "id"})

    merged = results.merge(sub_gt, on="id", how="left")
    merged["relevan"] = merged["relevan"].fillna(0).astype(int)

    row_eval = {
        "Query": query,
        "Kategori": kategori_query,
        "Retrieved": int(len(merged))
    }

    for k in K_VALUES:
        top_k = merged.head(k)
        relevant_k = int(top_k["relevan"].sum())
        precision_k = relevant_k / k

        row_eval[f"Relevan@{k}"] = relevant_k
        row_eval[f"P@{k}"] = round(precision_k, 4)

    eval_rows.append(row_eval)

eval_df = pd.DataFrame(eval_rows)

eval_path = os.path.join(eval_dir, "tabel2_precision_at_k.csv")
eval_df.to_csv(eval_path, index=False)

print("✅ Tabel 2 Precision@K tersimpan:", eval_path)
eval_df

✅ Tabel 2 Precision@K tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\tabel2_precision_at_k.csv


,Query,Kategori,Retrieved,Relevan@5,P@5,Relevan@10,P@10,Relevan@20,P@20
0,machine learning,Machine Learning,20,5,1.0,10,1.0,20,1.00
1,deep learning,Machine Learning,20,4,0.8,4,0.4,4,0.20
2,data mining,Machine Learning,20,0,0.0,0,0.0,0,0.00
3,website application,Web Application,20,0,0.0,0,0.0,0,0.00
4,web system,Web Application,20,0,0.0,0,0.0,0,0.00
5,web application,Web Application,20,4,0.8,4,0.4,4,0.20
6,cyber security,Cyber Security,20,5,1.0,10,1.0,20,1.00
7,network security,Cyber Security,20,0,0.0,0,0.0,0,0.00
8,software security,Cyber Security,20,0,0.0,0,0.0,0,0.00
9,mobile application,Mobile Application,20,1,0.2,1,0.1,1,0.05


In [80]:
# =========================================================
# CELL 12 - RATA-RATA PRECISION
# =========================================================
summary_rows = []

for k in K_VALUES:
    summary_rows.append({
        "Metric": f"Mean P@{k}",
        "Value": round(eval_df[f"P@{k}"].mean(), 4),
        "Percentage": round(eval_df[f"P@{k}"].mean() * 100, 2)
    })

summary_df = pd.DataFrame(summary_rows)

kat_avg_rows = []

for kategori, group in eval_df.groupby("Kategori"):
    row = {"Kategori": kategori}
    for k in K_VALUES:
        row[f"Rata-rata P@{k}"] = round(group[f"P@{k}"].mean(), 4)
    kat_avg_rows.append(row)

kat_avg_df = pd.DataFrame(kat_avg_rows)

summary_path = os.path.join(eval_dir, "tabel2_rata_rata_keseluruhan.csv")
kat_avg_path = os.path.join(eval_dir, "tabel2_rata_rata_per_kategori.csv")

summary_df.to_csv(summary_path, index=False)
kat_avg_df.to_csv(kat_avg_path, index=False)

print("✅ Rata-rata keseluruhan:")
print(summary_df.to_string(index=False))

print("\n✅ Rata-rata per kategori:")
kat_avg_df

✅ Rata-rata keseluruhan:
   Metric  Value  Percentage
 Mean P@5 0.3455       34.55
Mean P@10 0.2636       26.36
Mean P@20 0.2227       22.27

✅ Rata-rata per kategori:


,Kategori,Rata-rata P@5,Rata-rata P@10,Rata-rata P@20
0,Cyber Security,0.3333,0.3333,0.3333
1,Machine Learning,0.6000,0.4667,0.4000
2,Mobile Application,0.1000,0.0500,0.0250
3,Web Application,0.2667,0.1333,0.0667


In [81]:
# =========================================================
# CELL 13 - TABEL 3: JUMLAH KEMUNCULAN ARTIKEL
# =========================================================
kemunculan_df = pd.DataFrame([
    {
        "Judul Artikel": title,
        "Jumlah Kemunculan": count
    }
    for title, count in sorted(kemunculan_all.items(), key=lambda item: -item[1])
])

tabel3b_rows = []

for kategori in sorted(kemunculan_kat.keys()):
    data = kemunculan_kat[kategori]
    top_items = sorted(data.items(), key=lambda item: -item[1])[:10]

    for title, count in top_items:
        tabel3b_rows.append({
            "Kategori": kategori,
            "Judul Artikel": title,
            "Jumlah Kemunculan": count
        })

tabel3b_df = pd.DataFrame(tabel3b_rows)

kemunculan_path = os.path.join(eval_dir, "tabel3a_kemunculan_semua.csv")
kemunculan_kat_path = os.path.join(eval_dir, "tabel3b_kemunculan_per_kategori.csv")

kemunculan_df.to_csv(kemunculan_path, index=False)
tabel3b_df.to_csv(kemunculan_kat_path, index=False)

print("✅ Tabel kemunculan tersimpan")
kemunculan_df.head(10)

✅ Tabel kemunculan tersimpan


,Judul Artikel,Jumlah Kemunculan
0,A new mobile application of agricultural pests...,5
1,Early web application attack detection using n...,5
2,DeepCrop: Deep learning-based crop disease pre...,4
3,Deep learning methods for accurate skin cancer...,4
4,A systematic literature review on the cyber se...,4
5,A comprehensive review of cyber security vulne...,4
6,A hybrid fuzzy rule-based multi-criteria frame...,4
7,Design and implementation of smart hydroponics...,4
8,A multi-objective active learning platform and...,4
9,Machine learning based diabetes prediction and...,4


In [82]:
# =========================================================
# CELL 14 - SIMPAN OUTPUT EVALUASI
# =========================================================
os.makedirs(eval_dir, exist_ok=True)

tabel1_df.to_csv(
    os.path.join(eval_dir, "tabel1_hasil_pencarian.csv"),
    index=False
)

eval_df.to_csv(
    os.path.join(eval_dir, "tabel2_precision_at_k.csv"),
    index=False
)

summary_df.to_csv(
    os.path.join(eval_dir, "tabel2_rata_rata_keseluruhan.csv"),
    index=False
)

kat_avg_df.to_csv(
    os.path.join(eval_dir, "tabel2_rata_rata_per_kategori.csv"),
    index=False
)

kemunculan_df.to_csv(
    os.path.join(eval_dir, "tabel3a_kemunculan_semua.csv"),
    index=False
)

tabel3b_df.to_csv(
    os.path.join(eval_dir, "tabel3b_kemunculan_per_kategori.csv"),
    index=False
)

print("\n✅ Semua file tersimpan di data/evaluation/")
print(" - ground_truth_template.csv")
print(" - ground_truth_labeled.csv")
print(" - tabel1_hasil_pencarian.csv")
print(" - tabel2_precision_at_k.csv")
print(" - tabel2_rata_rata_keseluruhan.csv")
print(" - tabel2_rata_rata_per_kategori.csv")
print(" - tabel3a_kemunculan_semua.csv")
print(" - tabel3b_kemunculan_per_kategori.csv")


✅ Semua file tersimpan di data/evaluation/
 - ground_truth_template.csv
 - ground_truth_labeled.csv
 - tabel1_hasil_pencarian.csv
 - tabel2_precision_at_k.csv
 - tabel2_rata_rata_keseluruhan.csv
 - tabel2_rata_rata_per_kategori.csv
 - tabel3a_kemunculan_semua.csv
 - tabel3b_kemunculan_per_kategori.csv


In [83]:
# =========================================================
# CELL 15 - HELPER UPSERT SUPABASE
# =========================================================
def to_records_safe(dataframe):
    return dataframe.where(pd.notnull(dataframe), None).to_dict(orient="records")

def upsert_batches(table_name, records, on_conflict, batch_size=500):
    total = len(records)

    if total == 0:
        print(f"⚠️ Tidak ada data untuk disimpan ke {table_name}")
        return

    for start in range(0, total, batch_size):
        batch = records[start:start + batch_size]

        supabase.table(table_name).upsert(
            batch,
            on_conflict=on_conflict
        ).execute()

        print(f"  → Batch {start // batch_size + 1}: {len(batch)} data")

    print(f"✅ Upsert {total} data ke {table_name}")

In [84]:
# =========================================================
# CELL 16 - SIMPAN EVALUATION KE SUPABASE
# table: evaluation_precision_at_k
# K = 5, 10, 20 sesuai proposal
# =========================================================
ts = datetime.now(timezone.utc).isoformat()

rows_eval_db = []

for _, row in eval_df.iterrows():
    query = row["Query"]

    for k in K_VALUES:
        rows_eval_db.append({
            "compared_text": query,
            "k": int(k),
            "retrieved_count": int(row["Retrieved"]),
            "relevant_retrieved": int(row[f"Relevan@{k}"]),
            "precision_at_k": float(row[f"P@{k}"]),
            "updated_at": ts
        })

upsert_batches(
    table_name=EVAL_TABLE,
    records=rows_eval_db,
    on_conflict="compared_text,k",
    batch_size=500
)

  → Batch 1: 33 data
✅ Upsert 33 data ke evaluation_precision_at_k


In [85]:
# =========================================================
# CELL 17 - VALIDASI SUPABASE
# =========================================================
check = (
    supabase.table(EVAL_TABLE)
    .select("compared_text,k,precision_at_k", count="exact")
    .limit(40)
    .execute()
)

print("✅ Total row evaluation_precision_at_k:", check.count)
check.data

✅ Total row evaluation_precision_at_k: 36


[{'compared_text': 'machine learning', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'machine learning', 'k': 10, 'precision_at_k': 1},
 {'compared_text': 'machine learning', 'k': 20, 'precision_at_k': 1},
 {'compared_text': 'deep learning', 'k': 5, 'precision_at_k': 0.8},
 {'compared_text': 'deep learning', 'k': 10, 'precision_at_k': 0.4},
 {'compared_text': 'deep learning', 'k': 20, 'precision_at_k': 0.2},
 {'compared_text': 'data mining', 'k': 5, 'precision_at_k': 0},
 {'compared_text': 'data mining', 'k': 10, 'precision_at_k': 0},
 {'compared_text': 'data mining', 'k': 20, 'precision_at_k': 0},
 {'compared_text': 'website application', 'k': 5, 'precision_at_k': 0},
 {'compared_text': 'website application', 'k': 10, 'precision_at_k': 0},
 {'compared_text': 'web development', 'k': 20, 'precision_at_k': 0.05},
 {'compared_text': 'website application', 'k': 20, 'precision_at_k': 0},
 {'compared_text': 'web system', 'k': 5, 'precision_at_k': 0},
 {'compared_text': 'web system', 'k': 